## A workshop for developing weights for match ratings algorithms

For now, we will look at using the PCA algorithm

We will start with just looking at one position: CM

### Loading the data
First, we need to load the full matches dataset, defining the root folder and the path to the matches dataset.

In [1]:
# Loading the data
import json
import pandas as pd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from sklearn.covariance import MinCovDet
import numpy as np

# Define the project root (one folder up from /workshop)
project_root = Path("..").resolve()

# Add the project root to sys.path so Jupyter can find your 'src' module
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Load data/valencia_cf_1/matches.json
matches_path = project_root / "tests" / "fixtures" / "testing_data" / "data" / "valencia_cf_1" / "matches.json"
raw_matches = json.load(open(matches_path))
print(f"Loaded {len(raw_matches)} matches from {matches_path}")

Loaded 97 matches from C:\Users\kazik\Projects\Gaffers-Clipboard\tests\fixtures\testing_data\data\valencia_cf_1\matches.json


### Filtering the data
Now that we have the full matches dataset loaded, we can collect it into a pandas DataFrame.
In doing so, we focus on the player_performances section, keeping the match_id, half_length, and team-context fields (team_possession, team_xg, opponent_xg, team_passes).
The tracked team is assumed to be Valencia CF, so home/away stats are mapped into those team-context fields before filtering to CM-only performances.

In [2]:
normalized_df = pd.json_normalize(
    raw_matches,
    record_path="player_performances",
    meta=[
        "id", ["data", "half_length"], 
        ["data", "home_team_name"], 
        ["data", "away_team_name"], 
        ["data", "home_stats", "possession"], 
        ["data", "away_stats", "possession"],
        ["data", "home_stats", "xg"],
        ["data", "away_stats", "xg"],
        ["data", "home_stats", "passes"],
        ["data", "away_stats", "passes"]
    ]
)
# Create 4 new columns: team_possession, team_xg, opponent_xg, and team_passes, and set all the values to NaN
normalized_df["team_possession"] = float("nan")
normalized_df["team_xg"] = float("nan")
normalized_df["opponent_xg"] = float("nan")
normalized_df["team_passes"] = float("nan")

for index, performance in normalized_df.iterrows():
    # Check if the performance is for the home team or away team
    if performance["data.home_team_name"] == "Valencia CF":
        normalized_df.at[index, "team_possession"] = performance["data.home_stats.possession"]
        normalized_df.at[index, "team_xg"] = performance["data.home_stats.xg"]
        normalized_df.at[index, "opponent_xg"] = performance["data.away_stats.xg"]
        normalized_df.at[index, "team_passes"] = performance["data.home_stats.passes"]
    else:
        normalized_df.at[index, "team_possession"] = performance["data.away_stats.possession"]
        normalized_df.at[index, "team_xg"] = performance["data.away_stats.xg"]
        normalized_df.at[index, "opponent_xg"] = performance["data.home_stats.xg"]
        normalized_df.at[index, "team_passes"] = performance["data.away_stats.passes"]

# Remove the home and away names, possession, xg, and passes columns
normalized_df = normalized_df.drop(columns=["data.home_team_name", "data.away_team_name", "data.home_stats.possession", "data.away_stats.possession", "data.home_stats.xg", "data.away_stats.xg", "data.home_stats.passes", "data.away_stats.passes"])

normalized_df = normalized_df.rename(columns={"id": "match_id"})

normalized_df = normalized_df[normalized_df["performance_type"] != "GK"]

normalized_df = normalized_df.dropna(axis=1, how="all")

# Now we want to filter for players with only "CM" in "positions_played"
pos_df = normalized_df[normalized_df["positions_played"].apply(lambda x: x == ["CM"])]

# Now remove columns that are all NaN
pos_df = pos_df.dropna(axis=1, how="all")

# Remove "performance_type" and "positions_played" columns
pos_df = pos_df.drop(columns=["performance_type", "positions_played"])

# Move "match_id" to the front, and rename the id column to "performance_id"
cols = pos_df.columns.tolist()
cols.insert(0, cols.pop(cols.index("match_id")))
pos_df = pos_df[cols]
pos_df.head()

# Rename "data.half_length" to "half_length"
pos_df = pos_df.rename(columns={"data.half_length": "half_length"})

# Output number of rows and columns, and the first few rows of the resulting dataframe
print(f"Number of rows: {pos_df.shape[0]}")
print(f"Number of columns: {pos_df.shape[1]}")
print("Columns:")
print(pos_df.columns.tolist())
print("\n")

# Output the max and min of every column, to get a sense of the range of values
for col in pos_df.columns:
    if col in ["match_id", "player_id"]:
        continue
    if pos_df[col].dtype in ["int64", "float64"]:
        print(f"{col}: min={pos_df[col].min()}, max={pos_df[col].max()}")

Number of rows: 264
Number of columns: 24
Columns:
['match_id', 'player_id', 'goals', 'assists', 'shots', 'shot_accuracy', 'passes', 'pass_accuracy', 'dribbles', 'dribble_success_rate', 'tackles', 'tackle_success_rate', 'offsides', 'fouls_committed', 'possession_won', 'possession_lost', 'minutes_played', 'distance_covered', 'distance_sprinted', 'half_length', 'team_possession', 'team_xg', 'opponent_xg', 'team_passes']


goals: min=0.0, max=8.0
assists: min=0.0, max=8.0
shots: min=0.0, max=12.0
shot_accuracy: min=0.0, max=100.0
passes: min=0.0, max=55.0
pass_accuracy: min=0.0, max=100.0
dribbles: min=0.0, max=54.0
dribble_success_rate: min=0.0, max=100.0
tackles: min=0.0, max=12.0
tackle_success_rate: min=0.0, max=100.0
offsides: min=0.0, max=8.0
fouls_committed: min=0.0, max=8.0
possession_won: min=0.0, max=8.0
possession_lost: min=0.0, max=14.0
minutes_played: min=2.0, max=94.0
distance_covered: min=0.3, max=14.1
distance_sprinted: min=0.0, max=8.0
team_possession: min=39.0, max=71.0


### Fixing data types
Now that we have the data in a DataFrame, we need to fix the data types of the columns. We will convert all the columns that are not identifiers (like "match_id" and "player_id") to numeric data types. This is especially important when it comes to calculating xT, as we will need to perform mathematical operations on the stats.

In [3]:
# 1. Select every column EXCEPT your identifiers
cols_to_convert = pos_df.columns.drop(['match_id', 'player_id'])

# 2. Force them all to numeric in one swoop
pos_df[cols_to_convert] = pos_df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# Optional: If any weird JSON text like "N/A" got turned into a NaN, 
# you can safely fill them with 0s to keep the math from breaking later.
pos_df[cols_to_convert] = pos_df[cols_to_convert].fillna(0)

### Data pre-processing and normalizing
Now that we have the data in a DataFrame, we can start pre-processing it. A number of pre-processing steps are needed before we can apply PCA to the data.<br>

**Minutes played minimum**<br>
First, we will filter the data to only include performances where the player played at least 10 minutes. This is to ensure that we are only looking at performances where the player had a significant impact on the match, and to avoid skewing the PCA results with performances where the player only played a few minutes.<br>


In [4]:
# Step 1: Removing players with less than 10 minutes played
pos_df = pos_df[pos_df["minutes_played"] >= 10]

**Volume masks**<br>
We need to implement a volume mask. For each percentage stat, their associated volume attempted stat must reach a certain threshold for the percentage stat to be considered valid. If the volume attempted stat does not reach the threshold, we will replace the percentage stat with the mean percentage stat for that stat across all performances that do reach the threshold. This is to ensure that we are not skewing the PCA results with invalid percentage stats. The thresholds for the masks will be as follows:<br>
Pass accuracy - 5 attempted passes <br>
Shot accuracy - 2 attempted shots<br>
Dribble success rate - 3 attempted dribbles<br>
Tackle success rate - 2 attempted tackles<br>

In [5]:
# Step 1: Volume masking
# For each percentage stat, their associated volume attempted stat must reach a certain threshold for the percentage 
# stat to be considered valid. For example, if a player attempted 0 passes, their pass accuracy should not be considered valid,
# and should be set to the mean.
volume_perc_pairs = [
    ("passes", "pass_accuracy"),
    ("shots", "shot_accuracy"),
    ("dribbles", "dribble_success_rate"),
    ("tackles", "tackle_success_rate")
]
volume_masks = {
    "passes": 5,
    "shots": 2,
    "dribbles": 3,
    "tackles": 2
}

for volume_col, perc_col in volume_perc_pairs:
    # apply the mask
    valid = pos_df[pos_df[volume_col] >= volume_masks[volume_col]]
    mean_value = valid[perc_col].mean()
    pos_df[perc_col] = pos_df.apply(
        lambda r: r[perc_col] if r[volume_col] >= volume_masks[volume_col] else mean_value,
        axis=1
    )

**Possession adjustment**<br>
One problem with this dataset is that it is for a top team with high possession, which can inflate volume stats. That means a strong performance on a low-possession team may look average if we compare raw counts.<br>
To soften this bias, we scale volume stats with a square-root adjustment:<br>$$PAdj_{Attacking} = Raw \times \sqrt{\frac{50}{Team\_Possession}}$$<br>
for positive-possession stats (passes, dribbles, shots, possession lost), and:<br>$$PAdj_{Defensive} = Raw \times \sqrt{\frac{50}{100 - Team\_Possession}}$$<br>
for negative-possession stats (tackles, possession won, fouls committed).

In [6]:
# Passes mean and std before adjustment
print(f"Passes mean before adjustment: {pos_df['passes'].mean()}, std: {pos_df['passes'].std()}")

# Possession adjustment
# 1. The Softened Attacking & Distribution Tax
atk_cols = ["passes", "dribbles", "shots", "possession_lost"]
for col in atk_cols:
    # np.sqrt softens the penalty so extreme performances aren't crushed
    pos_df[col] = pos_df[col] * np.sqrt(50.0 / pos_df["team_possession"])

# 2. The Softened Defensive Tax
def_cols = ["tackles", "possession_won", "fouls_committed"]
for col in def_cols:
    pos_df[col] = pos_df[col] * np.sqrt(50.0 / (100.0 - pos_df["team_possession"]))
# print the passes mean and std after adjustment
print(f"Passes mean after adjustment: {pos_df['passes'].mean()}, std: {pos_df['passes'].std()}")

Passes mean before adjustment: 24.6171875, std: 13.728813892043004
Passes mean after adjustment: 23.31582241643759, std: 12.64371072689838


**Positional xG share**<br>
Goals and assists are not possession-adjusted. In this notebook we use fixed CM goal and assist shares to tune their influence in later scoring: $Share_{Goals}=0.10$ and $Share_{Assists}=0.15$.<br>
These constants are set directly (no dataset-derived share is computed here), and they act as calibrated priors for CM contributions rather than a dynamic, per-match xG mean.

In [7]:
# Overriding this to hardcode goal share at 10% and assist share at 15%, to give them more weight in the final rating
cm_goal_share = 0.1
cm_assist_share = 0.15

**Normalizing by half length and minutes played**<br>
We want volume stats to be comparable across half-length settings and uneven minutes. The code does this in three stages: standardize to a base half length, apply Bayesian smoothing, then convert to per-90 rates.<br>
1) Half-length standardization to $H_{base}=10$:<br>
$$X_{adj} = X_{raw} \times \left( \frac{H_{base}}{H_{match}} \right)$$
2) Bayesian smoothing (shrinkage) with $d=30$ dummy minutes. The goal is to reduce the volatility of small samples. Without smoothing, short cameos can produce extreme per-90s that dominate PCA. Smoothing blends the observed rate with a prior rate so the estimate is more stable and comparable across minutes.<br>
The smoothed rate is a weighted average:<br>
$$r_{smoothed} = \frac{M_{played}}{M_{played} + d} r_{obs} + \frac{d}{M_{played}+d} r_{prior}$$
- For rare stats (goals, assists, shots) we use $r_{prior}=0$ because the typical short spell does not imply scoring output.
- For volume stats we use $r_{prior}$ equal to the league average per-90 from the adjusted dataset, because we expect typical passing/tackling volume even in short appearances.<br>
**Example (rare stat):** 1 shot in 5 minutes gives $r_{obs}=18$ shots per 90. With $d=30$, $$r_{smoothed} = \frac{5}{35} \times 18 + \frac{30}{35} \times 0 \approx 2.57$$ shots per 90. The cameo is no longer treated like a full-match shooting binge.<br>
**Example (volume stat):** League passes $=40$ per 90, player makes 5 passes in 5 minutes ($r_{obs}=90$). With $d=30$, $$r_{smoothed} = \frac{5}{35} \times 90 + \frac{30}{35} \times 40 \approx 47.1$$ passes per 90, pulling the estimate toward a realistic baseline.<br>
3) Convert back to per-90 using the smoothed rate:<br>
$$X_{p90} = \frac{X_{adj} + r_{prior} \times \left( \frac{d}{90} \right)}{M_{played} + d} \times 90$$
This keeps small samples from overpowering the PCA while still letting full-match performances dominate.

In [8]:
# ---------------------------------------------------------
# Step 1: Standardize Absolute Volume (The Half-Length Fix)
# ---------------------------------------------------------
cum_columns = ["goals", "assists", "shots", "passes", "dribbles", "tackles", 
               "possession_won", "possession_lost", "fouls_committed", "offsides", 
               "distance_covered", "distance_sprinted"]

h_base = 10.0

for col in cum_columns:
    # We overwrite the raw stat to represent a standard 10-minute half game.
    # This ensures 5-min half games don't corrupt our dataset averages later.
    pos_df[col] = pos_df[col] * (h_base / pos_df["half_length"])


# ---------------------------------------------------------
# Step 2: Set Bayesian Smoothing Parameters
# ---------------------------------------------------------
dummy_minutes = 30.0  # Shrinkage anchor (requires ~a half of football to break away from mean)


# ---------------------------------------------------------
# Step 3: Apply Smoothed Per-90 Scaling
# ---------------------------------------------------------
# A. Rare Stats (Assume 0 for dummy minutes)
rare_cols = ["goals", "assists", "shots"]
for col in rare_cols:
    pos_df[f"{col}_p90"] = (pos_df[col] / (pos_df["minutes_played"] + dummy_minutes)) * 90.0


# B. Volume Stats (Assume dataset average for dummy minutes)
volume_cols = ["passes", "dribbles", "tackles", "possession_won", "possession_lost", 
               "fouls_committed", "offsides", "distance_covered", "distance_sprinted"]

for col in volume_cols:
    # 1. Calculate the true, half-length-adjusted dataset average Per 90
    league_avg_p90 = (pos_df[col].sum() / pos_df["minutes_played"].sum()) * 90.0
    
    # 2. Calculate the dummy stat allocation for 45 minutes
    dummy_stat = league_avg_p90 * (dummy_minutes / 90.0)
    
    # 3. Apply the final smoothed formula
    pos_df[f"{col}_p90"] = ((pos_df[col] + dummy_stat) / (pos_df["minutes_played"] + dummy_minutes)) * 90.0

# Print to verify
print(f"Passes mean after smoothing: {pos_df['passes_p90'].mean():.2f}, std: {pos_df['passes_p90'].std():.2f}")

Passes mean after smoothing: 33.46, std: 5.72


**Expected Threat**<br>
Expected Threat (xT) measures the likelihood of a player creating a goal based on their actions. True xT models require spatial tracking, which this dataset lacks, so we use a proxy tied to vertical progression and distribution.<br>
The proxy uses per-90 workrate ratios and passing volume/accuracy:<br>
$$Bonus_{xT} = 0.25 \times \left( \frac{D_{sprinted,p90}}{D_{covered,p90}} \right) \times \ln(Pass_{acc} \times Passes_{p90} + 1)$$
This rewards high-intensity running paired with secure, high-volume passing.

In [9]:
pos_df["xT_bonus_p90"] = 0.25 * (pos_df["distance_sprinted_p90"] / pos_df["distance_covered_p90"]) * np.log(pos_df["pass_accuracy"] * pos_df["passes_p90"] + 1)

**Handling long right tails**<br>
Several stats have heavy right tails (e.g., goals, assists). Log-transforming these before z-scoring reduces skew and makes the PCA inputs more stable. The transform is:<br>
$$X_{log} = \log(X_{final} + 1)$$
This does not assume perfect normality; it simply limits the influence of extreme outliers.

In [10]:
# Columns to log for PCA only; do NOT modify pos_df in-place
cols_to_log = ["goals_p90", "assists_p90", "shots_p90", "offsides_p90", "fouls_committed_p90", "possession_won_p90", "possession_lost_p90"]

**Z-score calculation**<br>
Finally, to turn the data into a format that can be used for PCA, we will calculate the z-scores for each stat. There are two different ways we will calculate the z-score, based on the stat in question:
 - For negative stats that we want to penalize (fouls committed, offsides, possession lost), we use an inverted z-score formula, where a higher raw stat value results in a lower z-score, as we want to penalize players for having high values in these stats. The formula for this is:<br>$$Z_{inverted} = \frac{\mu - X}{\sigma}$$
 - For all other stats, we use the standard z-score formula, where a higher raw stat value results in a higher z-score, as we want to reward players for having high values in these stats. The formula for this is:<br> $$Z = \frac{X - \mu}{\sigma}$$

In [11]:
negative_stats = ["fouls_committed_p90", "offsides_p90", "possession_lost_p90"]

# Build PCA dataframe from pos_df and apply log to long-tail columns (PCA-only)
pca_df = pos_df.copy()
pca_df[cols_to_log] = np.log(pca_df[cols_to_log] + 1)

# Define ordered stat list used for PCA (must match final weight order)
stat_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xT_bonus_p90"]
# Calculate z-scores on the log-transformed PCA dataframe
for col in stat_names:
    if col == "goals":
        mean = np.log(pos_df["team_xg"] * cm_goal_share + 1)
        std = np.log(pca_df[col]).std()
    elif col == "assists":
        mean = np.log(pos_df["team_xg"] * cm_assist_share + 1)
        std = np.log(pca_df[col]).std()
    else:
        mean = pca_df[col].mean()
        std = pca_df[col].std()
    if std == 0:
        pca_df[f"{col}_z"] = 0
    elif col in negative_stats:
        pca_df[f"{col}_z"] = (mean - pca_df[col]) / std
    else:
        pca_df[f"{col}_z"] = (pca_df[col] - mean) / std

# final matrix for PCA (z-scores from the log-transformed matrix)
final_cm_df = pca_df[[f"{col}_z" for col in stat_names]]

### Principal Component Analysis (PCA)
We run PCA on the z-scored features to discover the main directions of variation and to derive stable, interpretable weights for the CM position.<br><br>
**Block scaling (why and how)**<br>
The 17 stats are grouped into tactical blocks so that football roles are represented evenly:<br>
Attacking (goals_p90_z, assists_p90_z, shots_p90_z, shot_accuracy_z, offsides_p90_z), Passing (passes_p90_z, pass_accuracy_z), Dribbling (dribbles_p90_z, dribble_success_rate_z), Safety (possession_lost_p90_z), Defending (tackles_p90_z, tackle_success_rate_z, possession_won_p90_z, fouls_committed_p90_z), Workrate_and_Threat (distance_covered_p90_z, distance_sprinted_p90_z, xT_bonus_p90_z).<br><br>
Without block scaling, blocks with more features (e.g., Attacking has 5 stats) add more total variance purely because of size. If each stat had variance 1, a block with 5 stats would contribute about 5 units of variance, while a block with 2 stats would contribute about 2. That would bias PCA toward larger blocks regardless of football importance.<br>
We correct this by scaling each stat by $1/\sqrt{k_{block}}$. This makes the expected total variance per block comparable: $k \times (1/\sqrt{k})^2 = 1$. The Safety block has $k=1$, so it remains unchanged.<br>
**Example:** Attacking has $k=5$, so each stat is scaled by $1/\sqrt{5} \approx 0.447$. Passing has $k=2$, so each stat is scaled by $1/\sqrt{2} \approx 0.707$. This prevents Attacking from overpowering Passing just because it has more columns.<br><br>
**PCA mechanics (what it is doing)**<br>
PCA finds orthogonal directions (principal components) that capture maximal variance in the data. We first estimate a robust covariance matrix with Minimum Covariance Determinant (MCD, support_fraction=0.98) to reduce outlier influence. PCA then decomposes this covariance into eigenvalues $\lambda_i$ and eigenvectors $v_i$.<br>
- Each eigenvector $v_i$ is a component: a weighted combination of stats that captures a distinct pattern of performance.
- Each eigenvalue $\lambda_i$ is the variance explained by that component.
We sort components by $\lambda_i$ and keep $k=9$, which is the number needed to reach about 85% explained variance in this dataset.<br><br>
**How components translate to football patterns**<br>
Components often align with intuitive roles. For example, a component with large positive loadings on passes_p90_z and pass_accuracy_z represents distribution quality. Another with high loadings on tackles_p90_z and possession_won_p90_z represents defensive contribution. PCA discovers these patterns automatically from the covariance structure.<br><br>
**From components to weights**<br>
For each stat $j$, we aggregate its absolute loading across the retained components, scaled by $\sqrt{\lambda_i}$, then normalize:<br>
$$w_j = \frac{\sum_{i=1}^{k} |v_{j,i}| \sqrt{\lambda_i}}{\sum_{j} \sum_{i=1}^{k} |v_{j,i}| \sqrt{\lambda_i}}$$
We use absolute loadings so a stat contributes whether it loads positively or negatively to a component. Scaling by $\sqrt{\lambda_i}$ gives more weight to components that explain more variance. The final $w_j$ values sum to 1 and serve as the CM stat weights.

In [12]:
# 1. Define the "Blocks" of your Central Midfielder
blocks = {
    "Attacking": [
        "goals_p90_z", "assists_p90_z", "shots_p90_z", "shot_accuracy_z", "offsides_p90_z"
    ],
    "Passing": [
        "passes_p90_z", "pass_accuracy_z"
    ],
    "Dribbling": [
        "dribbles_p90_z", "dribble_success_rate_z"
    ],
    "Safety": [
        "possession_lost_p90_z",
    ],
    "Defending": [
        "tackles_p90_z", "tackle_success_rate_z", "possession_won_p90_z", "fouls_committed_p90_z"
    ],
    "Workrate_and_Threat": [
        "distance_covered_p90_z", "distance_sprinted_p90_z", "xT_bonus_p90_z"
    ]
}

# 2. Create a copy of your PCA dataframe so we don't mess up the original Z-scores
# Assume 'final_cm_df' is the dataframe containing just the 17 Z-score columns
scaled_pca_df = final_cm_df.copy()

# 3. Apply the Block Scaling math (Divide by the square root of the block size)
for stats in blocks.values():
    k = len(stats)  # Number of stats in this block
    scale_factor = np.sqrt(k)

    for stat in stats:
        # Scale the Z-score down
        scaled_pca_df[stat] = scaled_pca_df[stat] / scale_factor

# 4. Now run your robust PCA on the SCALED dataframe
mincovdet_random_state = 42

mcd = MinCovDet(random_state=mincovdet_random_state, support_fraction=0.98).fit(scaled_pca_df)

cov = mcd.covariance_

eigenvalues, eigenvectors = np.linalg.eigh(cov)

sorted_indicies = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_indicies]
eigenvectors = eigenvectors[:, sorted_indicies]

k = 9

top_eigenvectors = eigenvectors[:, :k]
top_variances = eigenvalues[:k]

weighted_loadings = np.abs(top_eigenvectors) * np.sqrt(top_variances)

raw_weights = np.sum(weighted_loadings, axis=1)

# Normalize to sum to 1
final_weights = raw_weights / np.sum(raw_weights)

stat_names = ["goals", "assists", "shots", "shot_accuracy", "passes", "pass_accuracy",
              "dribbles", "dribble_success_rate", "tackles", "tackle_success_rate", "offsides",
              "fouls_committed", "possession_won", "possession_lost", "distance_covered", "distance_sprinted", "xT_bonus"]

print("\nFinal weights (normalized to sum to 1):")
for stat, weight in zip(stat_names, final_weights):
    # print to 4 dp
    print(f"{stat}: {weight:.4f}")


Final weights (normalized to sum to 1):
goals: 0.0416
assists: 0.0435
shots: 0.0449
shot_accuracy: 0.0312
passes: 0.0963
pass_accuracy: 0.0858
dribbles: 0.0684
dribble_success_rate: 0.0830
tackles: 0.0575
tackle_success_rate: 0.0609
offsides: 0.0007
fouls_committed: 0.0587
possession_won: 0.0587
possession_lost: 0.0898
distance_covered: 0.0520
distance_sprinted: 0.0633
xT_bonus: 0.0636


### Output: CM stat weights
This cell writes the normalized PCA weights to the shared workshop/position_weights.json file under the CM section. Each weight aligns with the per-90 stat order in col_names and the weights sum to 1.

In [ ]:
# Outputting the final weights to a shared json file in the project root called position_weights.json
section_name = "CM"
col_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xT_bonus_p90"]
weights_dict = dict(zip(col_names, final_weights))
position_weights_path = project_root / "workshop" / "position_weights.json"

if position_weights_path.exists():
    with open(position_weights_path, "r") as f:
        position_weights = json.load(f)
else:
    position_weights = {}

position_weights[section_name] = weights_dict

with open(position_weights_path, "w") as f:
    json.dump(position_weights, f, indent=4)

### Output: Reference means and standard deviations
These means and standard deviations are computed from the fully preprocessed CM dataset (minutes filter, volume masking, possession adjustment, half-length normalization, Bayesian smoothing, per-90 conversion, and xT bonus).
They represent the baseline distribution the PCA saw, so later scoring can standardize new performances on the same scale.
In practice: compute the same per-90 features for a new match, then apply $z=(x-\mu)/\sigma$ (or the inverted form for negative stats). If any preprocessing step changes, regenerate this file.

In [ ]:
# Now lets look at storing the means and standard deviations used for the z-score calculations in a shared json file called position_means_stds.json
# Each position writes its baseline stats into its own section so the rating notebooks can read the same shared calibration data.
section_name = "CM"
means_stds = {}
for col in col_names:
    mean = pos_df[col].mean()
    std = pos_df[col].std()
    means_stds[col] = {"mean": mean, "std": std}
position_means_stds_path = project_root / "workshop" / "position_means_stds.json"

if position_means_stds_path.exists():
    with open(position_means_stds_path, "r") as f:
        position_means_stds = json.load(f)
else:
    position_means_stds = {}

position_means_stds[section_name] = means_stds

with open(position_means_stds_path, "w") as f:
    json.dump(position_means_stds, f, indent=4)